In [1]:

%load_ext autoreload
%autoreload 2

In [5]:
import os
import sys

ruta_proyecto = os.path.abspath("..")
if ruta_proyecto not in sys.path:
    sys.path.append(ruta_proyecto)

In [7]:

from torch.utils.data import DataLoader
from pathlib import Path
from utils.get_woof import ImagewoofColorizationDataset

import torch
import sys
from utils.trainer import trainer
DATA_DIR = Path("imagewoof2-160")
BATCH_SIZE = 8
EPOCHS = 5

train_ds = ImagewoofColorizationDataset(DATA_DIR, split="train")
val_ds   = ImagewoofColorizationDataset(DATA_DIR, split="val")

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE)

⚠️ No se encontraron imágenes en imagewoof2-160/train. Verificá la ruta y extensión de archivos.
⚠️ No se encontraron imágenes en imagewoof2-160/val. Verificá la ruta y extensión de archivos.


ValueError: num_samples should be a positive integer value, but got num_samples=0

In [ ]:
def train_model(model, train_loader, val_loader, save_name, criterion="l1"):
    save_path = "pesos_entrenados"
    model_path = Path(save_path) / save_name

    # carpeta donde guardamos las curvas loss vs epoch
    history_dir = Path("loss_vs_epoch")
    history_dir.mkdir(parents=True, exist_ok=True)
    history_path = history_dir / f"{save_name}_history.pt"

    train = False  # Cambia a True para forzar el reentrenamiento

    # Verificar si ya existe un modelo entrenado
    if model_path.exists() and not train:
        print(f"✅Modelo ya entrenado encontrado en '{model_path}'.")
        print("No se vuelve a entrenar para evitar sobreescritura.")

        # (opcional) si ya tenés la history guardada, podés devolverla:
        if history_path.exists():
            print(f"History encontrada en '{history_path}'.")
            history = torch.load(history_path, map_location="cpu")
            return history
        else:
            print("⚠️No se encontró history guardada para este modelo.")
            return None

    else:
        print("🚀No se encontró modelo entrenado, iniciando entrenamiento...")

        # ⬇️ahora trainer devuelve la history
        history = trainer(
            model,
            train_loader,
            val_loader,
            epochs=10,
            save_path=save_path,
            save_name=save_name,
            criterion=criterion,
        )

        print(f"💾Modelo guardado en: {model_path}")

        # ⬇️guardamos la history de forma GENERAL
        torch.save(history, history_path)
        print(f"📈History guardada en: {history_path}")

        return history

## Modelo con bacbone resnet, entrenado con L1 + Histograma

In [ ]:
import os

ruta_proyecto = os.path.abspath("..")
if ruta_proyecto not in sys.path:
    sys.path.append(ruta_proyecto)
from models.unet_resnet import get_model_unet_resnet34
from utils.trainer import train_model

model = get_model_unet_resnet34(pre_entrenado=True, congelar_encoder=False)
save_name = "unet_resnet34_histogram.pt"
train_model(model, train_loader, val_loader, save_name, criterion="histogram")

ModuleNotFoundError: No module named 'models.unet_resnet'